# EcoSort AI - Project-Trained Submission Edition

The final model is MobileNetV3Large initialized with ImageNet, followed by training on our audited project images. No SigLIP2 or image-text matching is used. Pretrained initialization is transfer learning, not a claim that we designed the base network.

## 1. Inspect the fixed dataset split

The raw pool contains 167,285 candidate images. Cleaning retained 147,303 images: 117,893 train, 14,698 validation and 14,712 test, across 17 classes. Only the training partition updates model weights. Validation chooses checkpoints and review settings; test measures performance. Never merge test images into training to increase the claimed sample count.

In [ ]:
from pathlib import Path
import json
import pandas as pd

assert Path('app.py').is_file(), 'Run this notebook from EcoSort-Demo.'
manifest = pd.read_csv('data/expanded/split_manifest.csv')
assert manifest.sha256.is_unique
assert manifest.groupby('group_id').split.nunique().max() == 1
print(manifest.groupby('split').size())
print(manifest.groupby('label').size())

## 2. Understand the five modules

app.py handles interaction. classifier.py loads our trained checkpoint. disposal_rules.py supplies conditional local guidance. train.py performs full training and fine-tuning. evaluate.py checks actual validation/test predictions and safety-related behavior. Training is a separate file because it is a distinct, essential project contribution.

## 3. Reproduce project-data training

train.py first trains a classification head over frozen ImageNet MobileNetV3 features, then fine-tunes the last 60 backbone layers except batch-normalization layers. Six fine-tuning epochs maximum visit every training image each epoch; early stopping can stop earlier. Class weights reduce imbalance; augmentation is training-only. Validation macro-F1 chooses the checkpoint. A new run must use a new output directory, and it never silently replaces the deployed model.

In [ ]:
import subprocess
import sys

RUN_TRAINING = False  # Set True only when ready for a full training run.
if RUN_TRAINING:
    subprocess.run([sys.executable, 'train.py', '--output', 'trained_reproduction'], check=True)
else:
    completed = json.loads(Path('trained_expanded/training_completed.json').read_text())
    print('Existing completed project training:', completed)

## 4. Re-evaluate without training on test images

Run evaluate.py to measure every validation and test image using the deployed checkpoint. Previously reported accuracy was 79.39% validation and 78.42% test; no 95% claim is made. These are previously observed dataset splits, not new independent evidence. SigLIP2's earlier web-photo results do not apply to this model.

In [ ]:
RUN_EVALUATION = False
if RUN_EVALUATION:
    subprocess.run([sys.executable, 'evaluate.py'], check=True)
record = json.loads(Path('evidence/trained_model_verified.json').read_text())
print(json.dumps(record['metrics'], indent=2))

## 5. Load and identify a photo

The training and application preprocessing both transpose EXIF orientation, convert to RGB, and resize to 224 x 224. Validation-selected temperature and review settings are applied. The model predicts material, not exact object identity. Do not infer a plastic bottle or recyclable container merely from the material label.

In [ ]:
from app import EcoSort, verify_release
from PIL import Image

verify_release()
engine = EcoSort()
photo_path = None  # Replace with Path('/absolute/path/to/photo.jpg').
if photo_path is not None:
    with Image.open(photo_path) as image:
        scores, decision, guidance = engine.classify(image)
    print(decision)
    print(guidance)

## 6. Keep disposal rules separate

Material alone is insufficient for a bin decision. User confirmation can distinguish an empty plastic bottle from a soft plastic bag. Batteries and other problem waste need special handling. City of Sydney service access and the contents or cleanliness of the item must still be checked.

In [ ]:
from disposal_rules import disposal_guidance

for item in ('Not confirmed', 'Empty plastic water/drink bottle', 'Soft plastic bag or wrapper'):
    print(item, '->', disposal_guidance('plastic', item)[0])

## 7. Launch the English upload/webcam interface

The submission uses our project-trained checkpoint with no zero-shot fallback. Runtime checks bind the deployed weights and source files to the evaluation record. Stop any old demo process before launching changed code.

In [ ]:
from app import launch

demo = launch(prevent_thread_lock=True)